In [0]:
# ==========================================
# Read Silver Table
# ==========================================

from pyspark.sql.functions import *
from pyspark.sql.types import *

silver_df = spark.table("silver_customer")

silver_df.printSchema()
silver_df.show(5)

# ==========================================
# Gold Collections Table
#Collections team only needs customers who still require collection efforts.
#Gold tables are business-specific. They don't have to include every column from Silver
# ==========================================


collections_df = silver_df.filter(
    col("AccountStatus").isin(
        "ACTIVE",
        "PENDING",
        "SUSPENDED",
        "CANCELLED"
    )
)

collections_df.show(10)

collections_df = collections_df.select(
    "CustomerID",
    "CustomerName",
    "ContactNumber",
    "State",
    "OutstandingBalance",
    "CollectionsPriority",
    "AccountStatus"
)

collections_df.show(10)

collections_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_collections")

# ==========================================
# Active Customers
# ==========================================

active_customer_df = silver_df.filter(
    col("AccountStatus") == "ACTIVE"
)

active_customer_df.show(10)

active_customer_df = active_customer_df.select(
    "CustomerID",
    "CustomerName",
    "ContactNumber",
    "State",
    "OutstandingBalance",
    "AccountStatus"
)

active_customer_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("gold_active_customers")

spark.sql("SHOW TABLES").show(truncate=False)







root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- PhoneNumbers: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- CreatedDate: date (nullable = true)
 |-- LastUpdatedDate: date (nullable = true)
 |-- ContactNumber: string (nullable = true)
 |-- State: string (nullable = true)
 |-- RejectReason: string (nullable = true)
 |-- TotalPayment: double (nullable = true)
 |-- OutstandingBalance: double (nullable = true)
 |-- AccountStatus: string (nullable = true)
 |-- CollectionsPriority: string (nullable = true)

+----------+----------------+--------------------+--------------------+-----------+---------------+-------------+-----+------------+------------------+------------------+-------------+-------------------+
|CustomerID|    CustomerName|        PhoneNumbers|             Address|CreatedDate|LastUpdatedDate|ContactNumber|State|RejectReason|      TotalPayment|OutstandingBalance|AccountStatus|CollectionsPriority|
+----------+